# SIGEX — Clustering Geográfico de Sondajes (DBSCAN)

Clustering de los 492 proyectos de sondaje SIGEX usando:
- **DBSCAN con distancia haversine** (eps = radio máximo en km)
- Clusters geográficamente acotados, sin chaining largo
- Objetivo: identificar distritos mineros potenciales de expansión


In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import haversine_distances
import folium
from folium.plugins import MarkerCluster
import warnings, os
warnings.filterwarnings("ignore")
np.random.seed(42)

BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
SIGEX_SHP = os.path.join(BASE_DIR, "../../01_Data/shared/ProyectosSIGEX/ProyectosSIGEX.shp")
OUT_DIR   = os.path.join(BASE_DIR, "outputs")
os.makedirs(OUT_DIR, exist_ok=True)
print("Librerías OK")


Librerías OK


## 1 — Carga y preprocesamiento

In [2]:
gdf = gpd.read_file(SIGEX_SHP)
print(f'Total proyectos: {len(gdf)}')
print(f'CRS: {gdf.crs}')
print(f'Columnas: {list(gdf.columns)}')

Total proyectos: 1097
CRS: EPSG:4326
Columnas: ['RUT', 'Entidad_In', 'Fecha_Ingr', 'Región', 'Identifica', 'Longitud', 'Latitud', 'Proyeccion', 'Recurso_Pr', 'Mapas_Geol', 'Levantamie', 'Levantam_1', 'Base_Datos', 'Levantam_2', 'Bases_Dato', 'Base_Dat_1', 'Bases_Da_1', 'Otros_Estu', 'Imagen_Sat', 'Estudios_A', 'Estudio_Es', 'Informe_Ge', 'Estudio__1', 'Presentaci', 'Plano_Mina', 'Topografí', 'Estimació', 'Propiedad_', 'Este_WGS84', 'Sur_WGS84', 'Huso_WGS84', 'Nombre_Pro', 'Enlace_Pro', 'Estado', 'Recurso__1', 'geometry']


In [3]:
# Extraer coordenadas desde geometry (ya en WGS84)
gdf['lat'] = gdf.geometry.y
gdf['lon'] = gdf.geometry.x

# Limpiar filas sin coordenadas válidas
df = gdf.dropna(subset=['lat', 'lon']).copy()
df = df[np.isfinite(df['lat']) & np.isfinite(df['lon'])].copy()
df = df.reset_index(drop=True)

# ── Clasificación por etapa (igual que build_dashboard_v2.py) ──────────────
def sigex_etapa(row):
    if (row.get('Estudio_Es','')=='ESPREF-001'
            or row.get('Plano_Mina','')=='PMIN-001'
            or row.get('Estimació','')=='REC-001'):
        return 'Factibilidad'
    if row.get('Base_Datos','')=='BDS-001':
        return 'Sondajes'
    if row.get('Levantam_2','')=='LEVGF-001':
        return 'Geofísica'
    if (row.get('Levantam_1','')=='BDGQ-001'
            or row.get('Mapas_Geol','')=='MGG-001'):
        return 'Prospección'
    return 'Exploración Inicial'

df['Etapa'] = df.apply(lambda r: sigex_etapa(r), axis=1)
print(f'Distribución por etapa:\n{df["Etapa"].value_counts()}')

# ── Filtrar solo SONDAJES ────────────────────────────────────────────────────
df = df[df['Etapa'] == 'Sondajes'].copy().reset_index(drop=True)
print(f'\nProyectos Sondajes: {len(df)}')
print(f'Distribución Recurso_Pr:\n{df["Recurso_Pr"].value_counts().head(10)}')
print(f'\nEstados:\n{df["Estado"].value_counts()}')
print(f'\nTop 10 empresas:\n{df["Entidad_In"].value_counts().head(10)}')


Distribución por etapa:
Etapa
Sondajes               492
Prospección            322
Geofísica              128
Exploración Inicial    120
Factibilidad            35
Name: count, dtype: int64

Proyectos Sondajes: 492
Distribución Recurso_Pr:
Recurso_Pr
Cu                251
Hidrocarburos      46
Cu-Mo              45
Cu-Au              30
Cu-Ag              18
Au-Ag              17
Fe                 12
Au                 10
Fe-Cu               8
Au-Ag-Cu-Pb-Zn      7
Name: count, dtype: int64

Estados:
Estado
Proyecto Aprobado         417
Proyecto en Evaluación     75
Name: count, dtype: int64

Top 10 empresas:
Entidad_In
CODELCO                                        44
ENAP                                           37
BHP CHILE INC                                  24
SQM S.A.                                       23
MINERA FREEPORT MCMORAN SOUTH AMERICA LTDA.    22
Compañía Minera San Gerónimo                   20
ANTOFAGASTA MINERALS S.A                       18
Anglo American Chile

In [4]:
# Columna empresa limpia — usar RUT como ID único de empresa
df['RUT_clean'] = df['RUT'].astype(str).str.strip()

# Normalizar region
def limpiar_region(x):
    x = str(x).upper()
    for pre in ['REGIÓN DE ', 'REGIÓN DEL ', 'REGION DE ', 'REGION DEL ']:
        x = x.replace(pre, '')
    return x.strip()

df['Region_Norm'] = df['Región'].apply(limpiar_region)

# Grupo recurso simplificado
def recurso_grupo(r):
    r = str(r)
    if r.startswith('Cu'):  return 'Cobre'
    if r.startswith('Au'):  return 'Oro'
    if 'Hidro' in r:        return 'Hidrocarburos'
    if r.startswith('Fe'):  return 'Hierro'
    return 'Otro'

df['Recurso_Grupo'] = df['Recurso_Pr'].apply(recurso_grupo)

print(df[['Nombre_Pro','Entidad_In','RUT_clean','lat','lon','Recurso_Grupo','Region_Norm','Estado']].head(5))

     Nombre_Pro                            Entidad_In   RUT_clean        lat  \
0  Punta Espora                                  ENAP  92604000-6 -52.477783   
1     Retamos-4                                  ENAP  92604000-6 -52.810913   
2     Retamos-5                                  ENAP  92604000-6 -52.819092   
3     Salvadora  Compañía Minera San Lorenzo Limitada  76645913-3 -26.374084   
4          Turi                        Terence Walker  14651619-K -22.222290   

         lon  Recurso_Grupo  Region_Norm                  Estado  
0 -69.461304  Hidrocarburos   MAGALLANES       Proyecto Aprobado  
1 -69.714600  Hidrocarburos   MAGALLANES       Proyecto Aprobado  
2 -69.708923  Hidrocarburos   MAGALLANES       Proyecto Aprobado  
3 -69.763794          Cobre      ATACAMA  Proyecto en Evaluación  
4 -68.306213          Cobre  ANTOFAGASTA       Proyecto Aprobado  


## 2 — Distancia haversine

In [5]:
# Distancia haversine entre proyectos (en km)
coords_rad = np.radians(df[["lat", "lon"]].values)
D_geo = haversine_distances(coords_rad) * 6371.0

print(f"D_geo shape: {D_geo.shape}")
print(f"Distancia máx Chile N-S: {D_geo.max():.0f} km")
# Distribución al vecino más cercano
nn = np.sort(D_geo + np.eye(len(D_geo))*99999, axis=1)[:,0]
for p in [25,50,75,90,95]:
    print(f"  NN p{p}: {np.percentile(nn,p):.1f} km")


D_geo shape: (492, 492)
Distancia máx Chile N-S: 3938 km
  NN p25: 1.5 km
  NN p50: 4.9 km
  NN p75: 12.0 km
  NN p90: 18.7 km
  NN p95: 25.8 km


In [6]:
# Con DBSCAN haversine directo no se necesita matriz precomputada
# El eps define el radio máximo de vecindad en km
EPS_KM = 10       # radio máximo: proyectos a >10 km no pueden clusterizarse
MIN_SAMPLES = 2   # mínimo de puntos para formar un cluster
print(f"EPS: {EPS_KM} km | MIN_SAMPLES: {MIN_SAMPLES}")


EPS: 10 km | MIN_SAMPLES: 2


## 3 — DBSCAN

In [7]:
# Efecto del eps sobre número de clusters y ruido
print("eps_km | min_s | n_cl | noise% | spread_p50 | spread_p90")
for eps in [5, 10, 15, 20]:
    for ms in [2, 3]:
        eps_rad = eps / 6371.0
        db = DBSCAN(eps=eps_rad, min_samples=ms, metric="haversine").fit(coords_rad)
        lbl = db.labels_
        n_cl = len(set(lbl)) - (1 if -1 in lbl else 0)
        noise = (lbl==-1).sum() / len(lbl) * 100
        spreads = []
        for c in set(lbl):
            if c < 0: continue
            idx = np.where(lbl==c)[0]
            if len(idx) >= 2: spreads.append(D_geo[np.ix_(idx,idx)].max())
        sp50 = float(np.median(spreads)) if spreads else 0
        sp90 = float(np.percentile(spreads,90)) if spreads else 0
        print(f"  {eps:5d} | {ms:5d} | {n_cl:4d} | {noise:5.1f}% | {sp50:6.1f} km | {sp90:.1f} km")


eps_km | min_s | n_cl | noise% | spread_p50 | spread_p90
      5 |     2 |   80 |  49.6% |    2.2 km | 6.8 km
      5 |     3 |   32 |  69.1% |    4.5 km | 9.5 km
     10 |     2 |   89 |  31.5% |    6.6 km | 15.5 km
     10 |     3 |   48 |  48.2% |    9.5 km | 18.0 km
     15 |     2 |   84 |  16.9% |   12.1 km | 28.0 km
     15 |     3 |   49 |  31.1% |   18.4 km | 44.9 km
     20 |     2 |   69 |   8.3% |   18.6 km | 47.1 km
     20 |     3 |   48 |  16.9% |   28.2 km | 55.8 km


In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

def clustering_metrics(labels, D_geo_full, coords_rad_full):
    """Calcula métricas de clustering para puntos asignados (excluye ruido=-1)."""
    mask = labels >= 0
    n_assigned = mask.sum()
    n_clusters = len(set(labels[mask]))
    
    if n_clusters < 2 or n_assigned < 10:
        return dict(Silhouette=np.nan, DB=np.nan, CH=np.nan)
    
    lbl_sub = labels[mask]
    
    # Silhouette con distancia haversine precomputada
    D_sub = D_geo_full[np.ix_(mask, mask)]
    # Normalizar a [0,1] para silhouette (no afecta ranking relativo)
    sil = silhouette_score(D_sub, lbl_sub, metric='precomputed')
    
    # DB y CH con coordenadas en km (lat/lon radianes × R_tierra)
    coords_km = coords_rad_full[mask] * 6371.0
    db  = davies_bouldin_score(coords_km, lbl_sub)
    ch  = calinski_harabasz_score(coords_km, lbl_sub)
    
    return dict(Silhouette=round(sil, 4), DB=round(db, 4), CH=round(ch, 1))

print(f"{'eps_km':>6} | {'min_s':>5} | {'n_cl':>5} | {'noise%':>7} | {'Silhouette':>10} | {'DB':>8} | {'CH':>10}")
print("-" * 70)

resultados_grid = []
for eps in [5, 10, 15, 20]:
    for ms in [2, 3]:
        eps_rad = eps / 6371.0
        db_obj  = DBSCAN(eps=eps_rad, min_samples=ms, metric="haversine").fit(coords_rad)
        lbl     = db_obj.labels_
        n_cl    = len(set(lbl)) - (1 if -1 in lbl else 0)
        noise_p = (lbl == -1).sum() / len(lbl) * 100
        
        mets = clustering_metrics(lbl, D_geo, coords_rad)
        row  = dict(eps_km=eps, min_samples=ms, n_clusters=n_cl,
                    noise_pct=round(noise_p,1), **mets)
        resultados_grid.append(row)
        
        print(f"{eps:>6} | {ms:>5} | {n_cl:>5} | {noise_p:>6.1f}% | "
              f"{mets['Silhouette']:>10.4f} | {mets['DB']:>8.4f} | {mets['CH']:>10.1f}")

df_grid_metrics = pd.DataFrame(resultados_grid)
print("\n✓ Mejor Silhouette:", df_grid_metrics.loc[df_grid_metrics['Silhouette'].idxmax(), ['eps_km','min_samples','Silhouette']].to_dict())
print("✓ Mejor DB (menor):", df_grid_metrics.loc[df_grid_metrics['DB'].idxmin(),  ['eps_km','min_samples','DB']].to_dict())
print("✓ Mejor CH (mayor):", df_grid_metrics.loc[df_grid_metrics['CH'].idxmax(),  ['eps_km','min_samples','CH']].to_dict())

In [ ]:
## ── Métricas del modelo final ─────────────────────────────────────────────
lbl_final = df['Cluster'].values
mask_final = lbl_final >= 0

# Métricas globales
mets_final = clustering_metrics(lbl_final, D_geo, coords_rad)
print("=" * 55)
print(f"  MÉTRICAS MODELO FINAL (eps={EPS_KM} km, min_s={MIN_SAMPLES})")
print("=" * 55)
print(f"  Clusters:           {n_cl}")
print(f"  Puntos asignados:   {mask_final.sum()} / {len(lbl_final)} ({mask_final.sum()/len(lbl_final)*100:.1f}%)")
print(f"  Ruido (noise):      {(~mask_final).sum()} ({(~mask_final).sum()/len(lbl_final)*100:.1f}%)")
print(f"  Silhouette Score:   {mets_final['Silhouette']:>8.4f}  (↑ mejor)")
print(f"  Davies-Bouldin:     {mets_final['DB']:>8.4f}  (↓ mejor)")
print(f"  Calinski-Harabasz:  {mets_final['CH']:>8.1f}  (↑ mejor)")
print("=" * 55)

# Silhouette por muestra (per-point) → identificar clusters problemáticos
lbl_sub    = lbl_final[mask_final]
D_sub      = D_geo[np.ix_(mask_final, mask_final)]
sil_vals   = silhouette_score(D_sub, lbl_sub, metric='precomputed')

# Silhouette por cluster (promedio de cada cluster)
from sklearn.metrics import silhouette_samples
sil_samples = silhouette_samples(D_sub, lbl_sub, metric='precomputed')
df_sil = pd.DataFrame({'Cluster': lbl_sub, 'sil': sil_samples})
sil_by_cl  = df_sil.groupby('Cluster')['sil'].agg(['mean','min','count']).reset_index()
sil_by_cl.columns = ['Cluster','Sil_mean','Sil_min','N']
sil_by_cl = sil_by_cl.sort_values('Sil_mean', ascending=False)

print(f"\nTop 10 clusters (mejor Silhouette interno):")
print(sil_by_cl.head(10).to_string(index=False))

print(f"\nBottom 10 clusters (peor Silhouette interno — más difusos):")
print(sil_by_cl.tail(10).to_string(index=False))

# Guardar métricas en CSV
df_grid_metrics.to_csv(os.path.join(OUT_DIR, 'sigex_clustering_metrics.csv'), index=False)
print(f"\nMétricas grid guardadas → {OUT_DIR}/sigex_clustering_metrics.csv")

## 3b — Métricas de Calidad del Clustering

- **Silhouette Score** [-1, 1]: mide cohesión interna vs separación entre clusters; más alto = mejor.
- **Davies-Bouldin (DB)**: ratio promedio de dispersión/separación; más bajo = mejor.
- **Calinski-Harabasz (CH)**: ratio varianza inter/intra cluster; más alto = mejor.
- **Noise%**: fracción de puntos no asignados (DBSCAN).
- Las métricas se calculan solo sobre puntos asignados (excluye ruido).

In [8]:
# Modelo final
eps_rad = EPS_KM / 6371.0
db = DBSCAN(eps=eps_rad, min_samples=MIN_SAMPLES, metric="haversine").fit(coords_rad)
df["Cluster"] = db.labels_
df["Prob"] = 1.0  # DBSCAN no tiene probabilidad, todos los asignados = 1

n_cl   = len(set(df["Cluster"])) - (1 if -1 in df["Cluster"].values else 0)
n_nois = int((df["Cluster"] == -1).sum())
print(f"Clusters: {n_cl}")
print(f"Ruido: {n_nois} ({n_nois/len(df)*100:.1f}%)")
print(f"Asignados: {len(df)-n_nois}")
# Spread por cluster
spreads_all = []
for c in set(df["Cluster"]):
    if c < 0: continue
    idx = np.where(df["Cluster"]==c)[0]
    if len(idx) >= 2: spreads_all.append(D_geo[np.ix_(idx,idx)].max())
print(f"Spread máx: {max(spreads_all):.1f} km | mediana: {np.median(spreads_all):.1f} km | p90: {np.percentile(spreads_all,90):.1f} km")


Clusters: 89
Ruido: 155 (31.5%)
Asignados: 337
Spread máx: 33.8 km | mediana: 6.6 km | p90: 15.5 km


In [9]:
# Resumen por cluster
resumen = df[df['Cluster'] >= 0].groupby('Cluster').agg(
    N=('Cluster', 'count'),
    lat_centroid=('lat', 'mean'),
    lon_centroid=('lon', 'mean'),
    Empresas=('Entidad_In', lambda x: x.nunique()),
    Empresa_top=('Entidad_In', lambda x: x.value_counts().index[0]),
    Recursos=('Recurso_Grupo', lambda x: x.value_counts().index[0]),
    Region_top=('Region_Norm', lambda x: x.value_counts().index[0]),
).reset_index()

# Dispersión geográfica dentro del cluster (km)
from scipy.spatial.distance import cdist
def cluster_spread_km(cluster_id):
    pts = df[df['Cluster'] == cluster_id][['lat','lon']].values
    if len(pts) < 2: return 0.0
    rad = np.radians(pts)
    d = haversine_distances(rad) * 6371.0
    return round(d.max(), 1)

resumen['spread_km'] = resumen['Cluster'].apply(cluster_spread_km)
print(resumen.sort_values('N', ascending=False).head(20).to_string(index=False))

 Cluster  N  lat_centroid  lon_centroid  Empresas                                 Empresa_top      Recursos  Region_top  spread_km
       1 23    -52.821186    -69.666378         1                                        ENAP Hidrocarburos  MAGALLANES       28.4
       7 19    -29.701097    -70.860472         3                Compañía Minera San Gerónimo         Cobre    COQUIMBO       21.6
       2 17    -26.340123    -69.653738         8                                     CODELCO         Cobre     ATACAMA       33.8
      86 10    -22.247456    -70.174760         2             Compañia Mantos de la Luna S.A.         Cobre ANTOFAGASTA       17.5
      31  9    -27.361300    -70.182508         7                 Compañía Minera Carmen Bajo         Cobre     ATACAMA       20.7
      12  9    -33.091946    -70.234714         1                                     CODELCO         Cobre  VALPARAÍSO       17.3
      10  8    -29.116552    -70.982698         4           Compañía Minera del Pac

## 4 — Guardar resultados

In [10]:
# CSV con todos los proyectos + cluster asignado
cols_out = ['Nombre_Pro','Entidad_In','RUT_clean','Recurso_Pr','Recurso_Grupo',
            'Región','Region_Norm','Estado','lat','lon','Cluster','Prob']
df[cols_out].to_csv(os.path.join(OUT_DIR, 'sigex_clusters.csv'), index=False)

# CSV resumen de clusters
resumen.to_csv(os.path.join(OUT_DIR, 'sigex_clusters_resumen.csv'), index=False)

print('CSVs guardados en', OUT_DIR)

CSVs guardados en /Users/mac/TrabajoTesis/FinalResultsFolder/02_Clustering/sigex_clustering/outputs


## 5 — Mapa interactivo Folium

In [11]:
import random

# Paleta de colores para clusters
random.seed(42)

def gen_colors(n):
    """Genera n colores hex distintos."""
    base = [
        '#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00',
        '#a65628','#f781bf','#999999','#66c2a5','#fc8d62',
        '#8da0cb','#e78ac3','#a6d854','#ffd92f','#e5c494',
        '#1f78b4','#b2df8a','#33a02c','#fb9a99','#e31a1c',
        '#fdbf6f','#ff7f00','#cab2d6','#6a3d9a','#ffff99',
        '#b15928','#8dd3c7','#ffffb3','#bebada','#fb8072'
    ]
    colors = base[:]
    while len(colors) < n:
        colors.append('#{:06x}'.format(random.randint(0, 0xFFFFFF)))
    return colors[:n]

n_cl_real = int(df['Cluster'].max()) + 1 if df['Cluster'].max() >= 0 else 0
palette   = gen_colors(n_cl_real)
NOISE_COLOR = '#cccccc'

def get_color(cluster_id):
    if cluster_id < 0:
        return NOISE_COLOR
    return palette[cluster_id % len(palette)]

# Colores por recurso (para ícono secundario)
RECURSO_ICON = {
    'Cobre':         'circle',
    'Oro':           'star',
    'Hidrocarburos': 'square',
    'Hierro':        'triangle',
    'Otro':          'circle'
}

print(f'Clusters a colorear: {n_cl_real}')

Clusters a colorear: 89


In [12]:
# Centro del mapa en Chile
m = folium.Map(
    location=[-27.0, -70.0],
    zoom_start=5,
    tiles='CartoDB positron'
)

# Capas: una por cluster + una para ruido
fg_clusters = folium.FeatureGroup(name='Clusters', show=True)
fg_noise    = folium.FeatureGroup(name='Ruido (no asignado)', show=False)

for _, row in df.iterrows():
    c   = int(row['Cluster'])
    col = get_color(c)
    label = f'Cluster {c}' if c >= 0 else 'Ruido'
    
    popup_html = f"""
    <b>{row['Nombre_Pro']}</b><br>
    Empresa: {row['Entidad_In']}<br>
    Recurso: {row['Recurso_Pr']}<br>
    Región: {row['Región']}<br>
    Estado: {row['Estado']}<br>
    <b>{label}</b> (prob={row['Prob']:.2f})
    """
    
    marker = folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5 if c >= 0 else 3,
        color=col,
        fill=True,
        fill_color=col,
        fill_opacity=0.8 if c >= 0 else 0.4,
        weight=1,
        popup=folium.Popup(popup_html, max_width=280),
        tooltip=f"{row['Nombre_Pro']} | {label}"
    )
    
    if c >= 0:
        fg_clusters.add_child(marker)
    else:
        fg_noise.add_child(marker)

# Centroides de clusters con label
fg_centroids = folium.FeatureGroup(name='Centroides', show=True)
for _, row in resumen.iterrows():
    c   = int(row['Cluster'])
    col = get_color(c)
    folium.Marker(
        location=[row['lat_centroid'], row['lon_centroid']],
        icon=folium.DivIcon(
            html=f'<div style="font-size:10px;font-weight:bold;color:{col};'
                 f'background:white;border:1px solid {col};padding:1px 3px;'
                 f'border-radius:3px;white-space:nowrap">C{c} (n={row["N"]})</div>',
            icon_size=(70, 20)
        ),
        tooltip=f"Cluster {c} | n={row['N']} | {row['Empresa_top']} | {row['Region_top']}"
    ).add_to(fg_centroids)

m.add_child(fg_clusters)
m.add_child(fg_noise)
m.add_child(fg_centroids)
folium.LayerControl(collapsed=False).add_to(m)

# Título
title_html = f'''
<div style="position:fixed;top:10px;left:50%;transform:translateX(-50%);
     background:white;padding:8px 16px;border-radius:6px;border:1px solid #aaa;
     z-index:9999;font-family:Arial;font-size:14px;font-weight:bold;">
     Proyectos SIGEX Sondajes — {n_cl_real} clusters | eps={EPS_KM} km | min_samples={MIN_SAMPLES}
</div>
'''
m.get_root().html.add_child(folium.Element(title_html))

map_path = os.path.join(OUT_DIR, 'sigex_clusters_map.html')
m.save(map_path)
print(f'Mapa guardado: {map_path}')
m

Mapa guardado: /Users/mac/TrabajoTesis/FinalResultsFolder/02_Clustering/sigex_clustering/outputs/sigex_clusters_map.html


## 6 — Análisis por empresa

In [13]:
# ¿Qué tan bien agrupan sus proyectos las grandes empresas?
top_empresas = df['Entidad_In'].value_counts().head(10).index
for emp in top_empresas:
    sub = df[df['Entidad_In'] == emp]
    n_total   = len(sub)
    n_en_cl   = int((sub['Cluster'] >= 0).sum())
    n_clusters = sub[sub['Cluster'] >= 0]['Cluster'].nunique()
    print(f"{emp[:40]:40s} | n={n_total:4d} | en_cluster={n_en_cl:3d} | clusters={n_clusters}")

CODELCO                                  | n=  44 | en_cluster= 34 | clusters=10
ENAP                                     | n=  37 | en_cluster= 35 | clusters=6
BHP CHILE INC                            | n=  24 | en_cluster=  8 | clusters=8
SQM S.A.                                 | n=  23 | en_cluster= 13 | clusters=7
MINERA FREEPORT MCMORAN SOUTH AMERICA LT | n=  22 | en_cluster= 17 | clusters=11
Compañía Minera San Gerónimo             | n=  20 | en_cluster= 18 | clusters=3
ANTOFAGASTA MINERALS S.A                 | n=  18 | en_cluster= 10 | clusters=8
Anglo American Chile Inversiones S.A.    | n=  14 | en_cluster=  9 | clusters=6
Compañia Mantos de la Luna S.A.          | n=  13 | en_cluster= 13 | clusters=3
Rio Tinto                                | n=  12 | en_cluster=  4 | clusters=3


In [14]:
# Clusters con mayor diversidad de empresas
print('Clusters con más empresas distintas:')
print(resumen.sort_values('Empresas', ascending=False)[['Cluster','N','Empresas','Empresa_top','Region_top','spread_km']].head(15).to_string(index=False))

Clusters con más empresas distintas:
 Cluster  N  Empresas                                            Empresa_top  Region_top  spread_km
       2 17         8                                                CODELCO     ATACAMA       33.8
      31  9         7                            Compañía Minera Carmen Bajo     ATACAMA       20.7
       3  7         6                                CIA. MINERA CARMEN BAJO     ATACAMA       13.5
      67  6         5                                    KUPRO RESOURCES SPA  VALPARAÍSO       16.7
      15  6         4                                Diomedes Cruz Solarzano    COQUIMBO        7.4
      71  6         4                        Compañia Mantos de la Luna S.A. ANTOFAGASTA       19.2
      10  8         4                      Compañía Minera del Pacífico S.A.     ATACAMA       11.2
      22  4         4                  Anglo American Chile Inversiones S.A. ANTOFAGASTA       11.7
      45  6         4                                 MINERA CA

In [15]:
# Distribución de tamaños de cluster
size_dist = resumen['N'].value_counts().sort_index()
print('Distribución de tamaños de cluster:')
print(size_dist.to_string())
print(f'\nMediana tamaño: {resumen["N"].median()}')
print(f'Percentil 90: {resumen["N"].quantile(0.9)}')

Distribución de tamaños de cluster:
N
2     41
3     19
4     15
5      1
6      4
7      2
8      1
9      2
10     1
17     1
19     1
23     1

Mediana tamaño: 3.0
Percentil 90: 6.200000000000003


In [16]:
# Distribución por región
print('Clusters por región (top región por cluster):')
print(resumen['Region_top'].value_counts().to_string())

Clusters por región (top región por cluster):
Region_top
ANTOFAGASTA             30
ATACAMA                 25
MAGALLANES               8
COQUIMBO                 8
TARAPACÁ                 7
VALPARAÍSO               6
ARICA Y PARINACOTA       2
REGIÓN AYSÉN             2
REGIÓN METROPOLITANA     1
